# Planning Pattern - ReAct Technique


Este agente proporciona formas para que el LLM divida una tarea en **subtareas más pequeñas y fáciles de lograr** sin perder de vista el objetivo final.

Este notebook está dividido en 3 partes principales:

1. **Inicialización**: En esta sección se importan todas las librerías necesarias y se inicializan las variables y herramientas que se utilizarán a lo largo del notebook. Es importante ejecutarlo todo tanto para la ejecución paso a paso como la automática.

2. **Ejecución paso a paso**: Aquí se realiza la ejecución del agente utilizando el bucle ReAct, paso a paso, para observar cómo el agente reflexiona, actúa y observa en cada iteración.

3. **Ejecución automática**: Finalmente, se ejecuta el agente de manera automática, delegando todo el proceso al agente para obtener el resultado final en una sola ejecución.

## 1. Inicialización

Comenzamos importando todas las librerías que utilizaremos en este tutorial, así como el cliente de Groq.

In [3]:
import sys
import os
import json
from openai import AzureOpenAI

# Añadir ruta para ejecución del código del src
folder_path = os.path.abspath(os.path.join(os.getcwd())) # Ajustar la ruta relativa
print(folder_path)
if folder_path not in sys.path:
    sys.path.append(folder_path)

from agentic_patterns_azure.tool_pattern.tool import tool
from agentic_patterns_azure.utils.extraction import extract_tag_content

c:\Users\david.vendrell.lopez\OneDrive - Accenture\Documents\Proyectos\template_POC\backend\src


### Ahora vamos a cargar la AZURE_OPENAI_APIKEY que tenemos guardada en el .env
Revisar que se haga el print correctamente de la API KEY que se haya puesto

In [4]:
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

# Now access the variable
AZURE_OPENAI_APIKEY = os.getenv("AZURE_OPENAI_APIKEY")
print(AZURE_OPENAI_APIKEY)

012f715c61ef4abdb958d96cb440c50a


### Nos conectamos al LLM de AZURE para hacer las llamadas a la IA

In [5]:
def conect_azure():
  client = AzureOpenAI(
    azure_endpoint = "https://llmcoeiberia-gpt4.openai.azure.com/",
    api_key=AZURE_OPENAI_APIKEY,  
    api_version="2024-02-01"
    )
  return(client)

model = "gpt-4-turbo"
client = conect_azure()

## A System Prompt for the ReAct Loop

Necesitamos un System Prompt para la técnica ReAct. Este describe el bucle ReAct, de manera que el LLM sea consciente de las tres operaciones que puede usar:

1. **Thought**: El LLM reflexionará sobre qué acción tomar.
2. **Action**: El LLM usará una Herramienta para "actuar sobre el entorno".
3. **Observation**: El LLM observará el resultado de la herramienta y reflexionará sobre el siguiente paso a realizar.

Vamos a encerrar todos los mensajes con etiquetas, como estas: <thought></thought>, <observation></observation>. Podríamos implementar la lógica ReAct sin estas etiquetas, pero para el LLM es mas sencillo entender las instrucciones de esta manera.



In [6]:
# Define the System Prompt as a constant
REACT_SYSTEM_PROMPT = """
You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don' make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
%s
</tools>

Example session:

<question>What's the current temperature in Madrid?</question>
<thought>I need to get the current weather in Madrid</thought>
<tool_call>{"name": "get_current_weather","arguments": {"location": "Madrid", "unit": "celsius"}, "id": 0}</tool_call>

You will be called again with this:

<observation>{0: {"temperature": 25, "unit": "celsius"}}</observation>

You then output:

<response>The current temperature in Madrid is 25 degrees Celsius</response>

Additional constraints:

- If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> tags.
"""

### Definimos las Tools

Estas son las funciones que puedes definir para que el agente tenga disponibles para usar durante la ejecución. Por ejemplo leer un csv, guardar un json, acceder a una API externa...

In [7]:
import math

@tool
def sum_two_elements(a: int, b: int) -> int:
    """
    Computes the sum of two integers.

    Args:
        a (int): The first integer to be summed.
        b (int): The second integer to be summed.

    Returns:
        int: The sum of `a` and `b`.
    """
    return a + b


@tool
def multiply_two_elements(a: int, b: int) -> int:
    """
    Multiplies two integers.

    Args:
        a (int): The first integer to multiply.
        b (int): The second integer to multiply.

    Returns:
        int: The product of `a` and `b`.
    """
    return a * b

@tool
def compute_log(x: int) -> float | str:
    """
    Computes the logarithm of an integer `x` with an optional base.

    Args:
        x (int): The integer value for which the logarithm is computed. Must be greater than 0.

    Returns:
        float: The logarithm of `x` to the specified `base`.
    """
    if x <= 0:
        return "Logarithm is undefined for values less than or equal to 0."
    
    return math.log(x)


### Adding the Tools signature to the System Prompt

Ahora, concatenamos las firmas de las tools y las añadimos al System Prompt. Aquí deberías reemplazarlo por las herramientas que hayas definido para tu proyecto siguiendo el mismo formato.

In [8]:
available_tools = {
    "sum_two_elements": sum_two_elements,
    "multiply_two_elements": multiply_two_elements,
    "compute_log": compute_log
}

In [9]:
tools_signature = sum_two_elements.fn_signature + ",\n" + multiply_two_elements.fn_signature + ",\n" + compute_log.fn_signature

In [10]:
print(tools_signature)

{"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}},
{"name": "multiply_two_elements", "description": "\n    Multiplies two integers.\n\n    Args:\n        a (int): The first integer to multiply.\n        b (int): The second integer to multiply.\n\n    Returns:\n        int: The product of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}},
{"name": "compute_log", "description": "\n    Computes the logarithm of an integer `x` with an optional base.\n\n    Args:\n        x (int): The integer value for which the logarithm is computed. Must be greater than 0.\n\n    Returns:\n        float: The logarithm of `x` to the specified `base`.\n    ", "parameters": {"prop

In [11]:
REACT_SYSTEM_PROMPT = REACT_SYSTEM_PROMPT % tools_signature

In [12]:
print(REACT_SYSTEM_PROMPT)


You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don' make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
{"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"pr

### User Prompt
Este prompt es donde le indicamos qué queremos y un poco los pasos a seguir para realizar la tarea con mayor éxito. Le especificamos también el formato del fichero de salida esperado

Debajo del todo, añadimos en el historial del chat estos prompts. Ya que en cada llamada que hacemos a la IA, le pasamos todo este historial para que sepa que debe hacer y todos los pasos que ya ha ejecutado previamente.

In [ ]:
USER_QUESTION = ''' I want to calculate the sum of 1234 and 5678 and multiply the result by 5. Then, I want to take the logarithm of this result '''

chat_history = [
    {
        "role": "system",
        "content": REACT_SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": f"<question>{USER_QUESTION}</question>"
    }
]

## 2. Ejecución paso a paso

### ReAct Loop Step 1

In [14]:
output = client.chat.completions.create(
    model="gpt-4-turbo",
    messages = chat_history,
    temperature=0,
    max_tokens=4000,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None
).choices[0].message.content

print(output)

<thought>First, I need to calculate the sum of 1234 and 5678. Then, I will multiply the result by 5. Finally, I will compute the logarithm of the final product.</thought>
<tool_call>
{"name": "sum_two_elements","arguments": {"a": 1234, "b": 5678}, "id": 1}
</tool_call>


In [ ]:
# Verificar si la respuesta contiene <tool_call>. En ese caso se ejecutará la función y se guardará en memoria
if "<tool_call>" in output:
    tool_call = extract_tag_content(output, tag="tool_call")
    tool_call = json.loads(tool_call.content[0])
    
    tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])
    print(tool_result)
    
    chat_history.append(
        {
            "role": "user",
            "content": f"<observation>{tool_result}</observation>"
        }
    )

# Si la respuesta contiene <observation>. En ese caso únicamente se guardará en memoria la respuesta.
elif "<observation>" in output:
    chat_history.append(
        {
            "role": "assistant",
            "content": output
        }
    )

<thought>First, I need to calculate the sum of 1234 and 5678. Then, I will multiply the result by 5. Finally, I will compute the logarithm of the final product.</thought>
<tool_call>
{"name": "sum_two_elements","arguments": {"a": 1234, "b": 5678}, "id": 1}
</tool_call>
6912


### ReAct Loop Step 2

In [18]:
output = client.chat.completions.create(
    model="gpt-4-turbo", # model = "deployment_name"
    messages = chat_history,
    temperature=0,
    max_tokens=4000,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None
).choices[0].message.content

print(output)

<tool_call>
{"name": "multiply_two_elements","arguments": {"a": 6912, "b": 5}, "id": 1}
</tool_call>


In [19]:
# Verificar si la respuesta contiene <tool_call>. En ese caso se ejecutará la función y se guardará en memoria
if "<tool_call>" in output:
    tool_call = extract_tag_content(output, tag="tool_call")
    tool_call = json.loads(tool_call.content[0])
    
    tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])
    print(tool_result)
    
    chat_history.append(
        {
            "role": "user",
            "content": f"<observation>{tool_result}</observation>"
        }
    )

# Si la respuesta contiene <observation>. En ese caso únicamente se guardará en memoria la respuesta.
elif "<observation>" in output:
    chat_history.append(
        {
            "role": "assistant",
            "content": output
        }
    )

34560


### ReAct Loop Step 3

In [20]:
output = client.chat.completions.create(
    model="gpt-4-turbo", # model = "deployment_name"
    messages = chat_history,
    temperature=0,
    max_tokens=4000,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None
).choices[0].message.content

print(output)

<tool_call>
{"name": "compute_log", "arguments": {"x": 34560}, "id": 0}
</tool_call>


In [21]:
# Verificar si la respuesta contiene <tool_call>. En ese caso se ejecutará la función y se guardará en memoria
if "<tool_call>" in output:
    tool_call = extract_tag_content(output, tag="tool_call")
    tool_call = json.loads(tool_call.content[0])
    
    tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])
    print(tool_result)
    
    chat_history.append(
        {
            "role": "user",
            "content": f"<observation>{tool_result}</observation>"
        }
    )

# Si la respuesta contiene <observation>. En ese caso únicamente se guardará en memoria la respuesta.
elif "<observation>" in output:
    chat_history.append(
        {
            "role": "assistant",
            "content": output
        }
    )

10.450452222917992


### ReAct Loop Step 4

In [22]:
output = client.chat.completions.create(
    model="gpt-4-turbo", # model = "deployment_name"
    messages = chat_history,
    temperature=0,
    max_tokens=4000,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None
).choices[0].message.content

print(output)

<response>The logarithm of the result is approximately 10.450452222917992</response>


In [23]:
# Verificar si la respuesta contiene <tool_call>. En ese caso se ejecutará la función y se guardará en memoria
if "<tool_call>" in output:
    tool_call = extract_tag_content(output, tag="tool_call")
    tool_call = json.loads(tool_call.content[0])
    
    tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])
    print(tool_result)
    
    chat_history.append(
        {
            "role": "user",
            "content": f"<observation>{tool_result}</observation>"
        }
    )

# Si la respuesta contiene <observation>. En ese caso únicamente se guardará en memoria la respuesta.
elif "<observation>" in output:
    chat_history.append(
        {
            "role": "assistant",
            "content": output
        }
    )

### ReAct Loop Step 5

In [24]:
output = client.chat.completions.create(
    model="gpt-4-turbo", # model = "deployment_name"
    messages = chat_history,
    temperature=0,
    max_tokens=4000,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None
).choices[0].message.content

print(output)

<response>The logarithm of the result is approximately 10.450452222917992</response>


In [25]:
# Verificar si la respuesta contiene <tool_call>. En ese caso se ejecutará la función y se guardará en memoria
if "<tool_call>" in output:
    tool_call = extract_tag_content(output, tag="tool_call")
    tool_call = json.loads(tool_call.content[0])
    
    tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])
    print(tool_result)
    
    chat_history.append(
        {
            "role": "user",
            "content": f"<observation>{tool_result}</observation>"
        }
    )

# Si la respuesta contiene <observation>. En ese caso únicamente se guardará en memoria la respuesta.
elif "<observation>" in output:
    chat_history.append(
        {
            "role": "assistant",
            "content": output
        }
    )

## 3. Ejecución automática

Ahora se ejecutarán de manera automática todas las iteraciones del bucle ReAct hasta obtener el resultado final, gracias al uso de la librería `agentic_patterns`. Esto simplifica el proceso al delegar la lógica de las iteraciones al agente, evitando la necesidad de manejar manualmente cada paso del bucle.

In [13]:
from agentic_patterns_azure.planning_pattern.react_agent import ReactAgent

Añade aqui el nombre de las tools definidas previamente. También, tendremos que poner en react_agent.py (linea 19) el mismo SYSTEM_PROMPT que hayamos definido en el notebook.

In [14]:
agent = ReactAgent(tools=[sum_two_elements, multiply_two_elements, compute_log])

Aqui pasamos como parametro USER_QUESTION, que es el prompt definido previamente

In [17]:
agent.run(user_msg=USER_QUESTION)


Thought: First, I need to calculate the sum of 1234 and 5678. Then, I will multiply that sum by 5. Finally, I will compute the logarithm of the resulting product.

Using Tool: sum_two_elements

Tool call dict: 
{'name': 'sum_two_elements', 'arguments': {'a': 1234, 'b': 5678}, 'id': 0}

Tool result: 
6912

Observations: {0: 6912}

Thought: Now that I have the sum of 1234 and 5678 which is 6912, I need to multiply this result by 5.

Using Tool: multiply_two_elements

Tool call dict: 
{'name': 'multiply_two_elements', 'arguments': {'a': 6912, 'b': 5}, 'id': 1}

Tool result: 
34560

Observations: {1: 34560}

Thought: Having obtained the product of 6912 and 5, which is 34560, the next step is to compute the logarithm of this result.

Using Tool: compute_log

Tool call dict: 
{'name': 'compute_log', 'arguments': {'x': 34560}, 'id': 2}

Tool result: 
10.450452222917992

Observations: {2: 10.450452222917992}


'The logarithm of the result is approximately 10.450452222917992'

---

We did it!! A ReAct Agent working as expected, completely from Scratch! 🚀🚀🚀🚀